<a href="https://colab.research.google.com/github/FelipeT1/PS-2026.1/blob/main/Projeto_PS_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bibliotecas usadas

import pandas as pd
import geopandas as gpd

# Introdução

#

## Coleta e limpeza de dados

Dois conjuntos de dados que se relacionam pela coluna shape_id queremos isso para saber onde cada ponto de cada linha e descobrir em que CISP ela passa.

*   SMTR/gtfs/shapes_geom.csv
*   Data-rio:
https://www.data.rio/datasets/PCRJ::itiner%C3%A1rios-de-servi%C3%A7os-de-%C3%B4nibus-regulares/explore?showTable=true



### TAREFA #1 EDA fora de ordem



In [ ]:
unique_itinerarios_shapes = set(itinerarios_df['shape_id'].unique())
unique_geom_shapes = set(shape_geom_df['shape_id'].unique())

# Shape IDs em itinerarios_df mas não em shape_geom_df
shapes_only_in_itinerarios = unique_itinerarios_shapes.difference(unique_geom_shapes)
print(f"Shape IDs apenas em 'itinerarios_df': {len(shapes_only_in_itinerarios)} IDs.\nExemplo de alguns: {list(shapes_only_in_itinerarios)[:5]}\n")

# Shape IDs em shape_geom_df mas não em itinerarios_df
shapes_only_in_geom = unique_geom_shapes.difference(unique_itinerarios_shapes)
print(f"Shape IDs apenas em 'shape_geom_df': {len(shapes_only_in_geom)} IDs.\nExemplo de alguns: {list(shapes_only_in_geom)[:5]}")

NameError: name 'itinerarios_df' is not defined

Existem shapes_ids que por algum motivo são exclusivos do dataset de itinerarios.

In [ ]:
shape_e_itinerarios_df['diff_length'] = abs(shape_e_itinerarios_df['shape_distance'] - shape_e_itinerarios_df['SHAPE__Length'])
shape_e_itinerarios_df['percent_diff_length'] = (shape_e_itinerarios_df['diff_length'] / ((shape_e_itinerarios_df['shape_distance'] + shape_e_itinerarios_df['SHAPE__Length']) / 2)) * 100

print("Estatísticas descritivas para a diferença absoluta de comprimento:")
print(shape_e_itinerarios_df['diff_length'].describe())
print("\nEstatísticas descritivas para a diferença percentual de comprimento:")
print(shape_e_itinerarios_df['percent_diff_length'].describe())

print("\nTop 5 linhas com maiores diferenças percentuais:")
print(shape_e_itinerarios_df.nlargest(5, 'percent_diff_length')[['shape_id', 'SHAPE__Length', 'shape_distance', 'diff_length', 'percent_diff_length']])

Estatísticas descritivas para a diferença absoluta de comprimento:
count     9478.000000
mean       435.954092
std       1750.603568
min          0.019476
25%         12.068901
50%         24.946457
75%         55.507171
max      24804.725240
Name: diff_length, dtype: float64

Estatísticas descritivas para a diferença percentual de comprimento:
count    9478.000000
mean        2.253537
std         9.410265
min         0.000054
25%         0.059482
50%         0.111756
75%         0.229018
max       110.028726
Name: percent_diff_length, dtype: float64

Top 5 linhas com maiores diferenças percentuais:
    shape_id  SHAPE__Length  shape_distance   diff_length  percent_diff_length
545     2bk6   16486.044675          4784.3  11701.744675           110.028726
546     2bk6   16486.044675          4784.3  11701.744675           110.028726
548     2bk6   16486.044675          4784.3  11701.744675           110.028726
549     2bk6   16486.044675          4784.3  11701.744675           110.02872

Para verificar se os shapes_id são de fato os mesmos em ambos os dataset foi verificada a distância contida nos dois datasets. Espera-se que se forem os mesmos tenham a mesma distância ou parecidas.

In [ ]:
shape_geom_df.head()

,feed_version,feed_start_date,feed_end_date,shape_id,shape,shape_distance,start_pt,end_pt,versao_modelo
0,2023-12-16,2023-12-16,2023-12-20,1oge,"LINESTRING(-43.232036 -22.923777, -43.23146 -2...",24466.1,POINT(-43.232036 -22.923777),POINT(-43.36544 -23.0016),201d79faee763526a030ff998bebea9782efe961
1,2025-06-01,2025-06-01,2025-07-15,9ptt,"MULTILINESTRING((-43.29031 -22.90076, -43.2902...",17517.0,POINT(-43.29031 -22.90076),POINT(-43.17068 -22.90907),02716ed5a495ae53c1c4948184b5d093aa3126a8
2,2025-11-08,2025-11-08,2025-11-18,35bu,"MULTILINESTRING((-43.17754 -22.90107, -43.1775...",33079.0,POINT(-43.17754 -22.90107),POINT(-43.39397 -22.95665),2a53b9c61f4b76fe07761a33083c714f9c35bcd0
3,2025-10-04,2025-10-04,2025-10-11,iz18,"LINESTRING(-43.19987 -22.93997, -43.19975 -22....",12591.1,POINT(-43.19987 -22.93997),POINT(-43.2238 -22.98097),40c2bc268bdc2e22bc2474d99bca38dcb9ea4c40
4,2024-03-11,2024-03-11,2024-03-17,2ibq,"LINESTRING(-43.621884 -22.968783, -43.62189 -2...",24430.2,POINT(-43.621884 -22.968783),POINT(-43.4637 -22.87668),201d79faee763526a030ff998bebea9782efe961


### Tarefa #1: Importar e juntar as tabelas CSV e GeoSJON das linhas pelo ShapeID

In [ ]:
itinerarios_df = pd.read_csv("/content/Itinerários_de_Serviços_de_Ônibus_Regulares.csv")

In [ ]:
itinerarios_df.head()

,fid,extensao,consorcio,descricao_desvio,tipo_rota,shape_id,direcao,destino,servico,SHAPE__Length,tipo_dia
0,1,24049,Intersul,NaN,regular,006j,0,Circular,LECD130,24049.649573,U
1,2,8502,Santa Cruz,NaN,regular,01gk,1,Jardim Bangu,819,8502.547401,U
2,3,12723,Intersul,NaN,regular,01o6,0,Circular,010,12723.322492,U
3,4,29403,Internorte,NaN,regular,01vm,0,Copacabana,SN457,29403.611996,U
4,5,25967,Santa Cruz,NaN,regular,040z,1,Terminal Campo Grande,857,25967.952125,U


In [ ]:
itinerarios_df[itinerarios_df['servico'] == '905']

,fid,extensao,consorcio,descricao_desvio,tipo_rota,shape_id,direcao,destino,servico,SHAPE__Length,tipo_dia
345,346,16214,Internorte,NaN,regular,O0905AAA0AIDU01,0,Irajá,905,16214.369748,U
346,347,17446,Internorte,NaN,regular,O0905AAA0AVDU01,1,Bonsucesso,905,17446.526660,U


In [ ]:
shape_geom_df = pd.read_csv("/content/shapes_geom.csv")

In [ ]:
shape_geom_df.head()

,feed_version,feed_start_date,feed_end_date,shape_id,shape,shape_distance,start_pt,end_pt,versao_modelo
0,2023-12-16,2023-12-16,2023-12-20,1oge,"LINESTRING(-43.232036 -22.923777, -43.23146 -2...",24466.1,POINT(-43.232036 -22.923777),POINT(-43.36544 -23.0016),201d79faee763526a030ff998bebea9782efe961
1,2025-06-01,2025-06-01,2025-07-15,9ptt,"MULTILINESTRING((-43.29031 -22.90076, -43.2902...",17517.0,POINT(-43.29031 -22.90076),POINT(-43.17068 -22.90907),02716ed5a495ae53c1c4948184b5d093aa3126a8
2,2025-11-08,2025-11-08,2025-11-18,35bu,"MULTILINESTRING((-43.17754 -22.90107, -43.1775...",33079.0,POINT(-43.17754 -22.90107),POINT(-43.39397 -22.95665),2a53b9c61f4b76fe07761a33083c714f9c35bcd0
3,2025-10-04,2025-10-04,2025-10-11,iz18,"LINESTRING(-43.19987 -22.93997, -43.19975 -22....",12591.1,POINT(-43.19987 -22.93997),POINT(-43.2238 -22.98097),40c2bc268bdc2e22bc2474d99bca38dcb9ea4c40
4,2024-03-11,2024-03-11,2024-03-17,2ibq,"LINESTRING(-43.621884 -22.968783, -43.62189 -2...",24430.2,POINT(-43.621884 -22.968783),POINT(-43.4637 -22.87668),201d79faee763526a030ff998bebea9782efe961


In [ ]:

teste_df = pd.merge(itinerarios_df, shape_geom_df, on='shape_id', how='left')

In [ ]:
teste_df[teste_df['servico'] == '857']

,fid,extensao,consorcio,descricao_desvio,tipo_rota,shape_id,direcao,destino,servico,SHAPE__Length,...,feed_version,feed_start_date,feed_end_date,shape,shape_distance,start_pt,end_pt,versao_modelo,geometry,geometry_obj
40,5,25967,Santa Cruz,NaN,regular,040z,1,Terminal Campo Grande,857,25967.952125,...,2024-12-31,2024-12-31,2025-01-01,"MULTILINESTRING((-43.64759 -22.96519, -43.6475...",25862.5,POINT(-43.64759 -22.96519),POINT(-43.55505 -22.90175),3e1630ddb15dca783e340faf23f08f68733f4813,"MULTILINESTRING ((-43.64759 -22.96519, -43.647...","MULTILINESTRING ((-43.64759 -22.96519, -43.647..."
41,5,25967,Santa Cruz,NaN,regular,040z,1,Terminal Campo Grande,857,25967.952125,...,2024-11-16,2024-11-16,2024-12-13,"MULTILINESTRING((-43.64759 -22.96519, -43.6475...",25862.5,POINT(-43.64759 -22.96519),POINT(-43.55505 -22.90175),c19231dde67a8732f822cb0f1a4ce9825a0360c0,"MULTILINESTRING ((-43.64759 -22.96519, -43.647...","MULTILINESTRING ((-43.64759 -22.96519, -43.647..."
42,5,25967,Santa Cruz,NaN,regular,040z,1,Terminal Campo Grande,857,25967.952125,...,2025-05-24,2025-05-24,2025-05-24,"MULTILINESTRING((-43.64759 -22.96519, -43.6475...",25862.5,POINT(-43.64759 -22.96519),POINT(-43.55505 -22.90175),bd3bdcb8b2b2c88ae81bcbe03f64ed235cc693d6,"MULTILINESTRING ((-43.64759 -22.96519, -43.647...","MULTILINESTRING ((-43.64759 -22.96519, -43.647..."
43,5,25967,Santa Cruz,NaN,regular,040z,1,Terminal Campo Grande,857,25967.952125,...,2025-04-18,2025-04-18,2025-04-30,"MULTILINESTRING((-43.64759 -22.96519, -43.6475...",25862.5,POINT(-43.64759 -22.96519),POINT(-43.55505 -22.90175),122abe77b87c75b8505fc679ef662556336ca16f,"MULTILINESTRING ((-43.64759 -22.96519, -43.647...","MULTILINESTRING ((-43.64759 -22.96519, -43.647..."
44,5,25967,Santa Cruz,NaN,regular,040z,1,Terminal Campo Grande,857,25967.952125,...,2026-02-02,2026-02-02,2026-02-10,"MULTILINESTRING((-43.64759 -22.96519, -43.6475...",25862.5,POINT(-43.64759 -22.96519),POINT(-43.55505 -22.90175),f7495a359559836032529b2d93bc4eff1a079265,"MULTILINESTRING ((-43.64759 -22.96519, -43.647...","MULTILINESTRING ((-43.64759 -22.96519, -43.647..."
6026,597,27461,Santa Cruz,NaN,regular,kb6o,0,Terminal Pingo D'Água,857,27461.173116,...,2024-05-03,2024-05-03,2024-05-14,"MULTILINESTRING((-43.55505 -22.90175, -43.5554...",28681.3,POINT(-43.55505 -22.90175),POINT(-43.64978 -22.9659),7ee9b89e4c11194700ba392d52b35401f8e01ed1,"MULTILINESTRING ((-43.55505 -22.90175, -43.555...","MULTILINESTRING ((-43.55505 -22.90175, -43.555..."
6027,597,27461,Santa Cruz,NaN,regular,kb6o,0,Terminal Pingo D'Água,857,27461.173116,...,2024-11-06,2024-11-06,2024-11-15,"MULTILINESTRING((-43.55505 -22.90175, -43.5554...",29030.0,POINT(-43.55505 -22.90175),POINT(-43.64759 -22.96519),b9ce9539b00b6d27a190f7a9370ae791676ec0d5,"MULTILINESTRING ((-43.55505 -22.90175, -43.555...","MULTILINESTRING ((-43.55505 -22.90175, -43.555..."
6028,597,27461,Santa Cruz,NaN,regular,kb6o,0,Terminal Pingo D'Água,857,27461.173116,...,2026-01-03,2026-01-03,2026-01-18,"MULTILINESTRING((-43.55505 -22.90175, -43.5554...",27498.6,POINT(-43.55505 -22.90175),POINT(-43.64759 -22.96519),8df7635a5781e2e43ecc1920667b18f29516caa0,"MULTILINESTRING ((-43.55505 -22.90175, -43.555...","MULTILINESTRING ((-43.55505 -22.90175, -43.555..."
6029,597,27461,Santa Cruz,NaN,regular,kb6o,0,Terminal Pingo D'Água,857,27461.173116,...,2025-02-28,2025-02-28,2025-03-14,"MULTILINESTRING((-43.55505 -22.90175, -43.5554...",27498.6,POINT(-43.55505 -22.90175),POINT(-43.64759 -22.96519),ca58bf94f33ad4121c10e7216999df32dddc36eb,"MULTILINESTRING ((-43.55505 -22.90175, -43.555...","MULTILINESTRING ((-43.55505 -22.90175, -43.555..."
6030,597,27461,Santa Cruz,NaN,regular,kb6o,0,Terminal Pingo D'Água,857,27461.173116,...,2024-09-13,2024-09-13,2024-09-28,"MULTILINESTRING((-43.55505 -22.90175, -43.5554...",28205.6,POINT(-43.55505 -22.90175),POINT(-43.647593 -22.964987),5d6ba3193f5d93ba8aa3bb4277ef03ccaac50665,"MULTILINESTRING ((-43.55505 -22.90175, -43.555...","MULTILINESTRING ((-43.55505 -22.90175, -43.555..."


In [ ]:
# Inner join pelo shape_id contido em ambas tabelas
shape_e_itinerarios_df = pd.merge(itinerarios_df, shape_geom_df, on='shape_id', how='inner')

In [ ]:
shape_e_itinerarios_df

,fid,extensao,consorcio,descricao_desvio,tipo_rota,shape_id,direcao,destino,servico,SHAPE__Length,tipo_dia,feed_version,feed_start_date,feed_end_date,shape,shape_distance,start_pt,end_pt,versao_modelo
0,1,24049,Intersul,NaN,regular,006j,0,Circular,LECD130,24049.649573,U,2026-03-12,2026-03-12,2026-03-13,"LINESTRING(-43.21592 -22.9801, -43.21543 -22.9...",24111.3,POINT(-43.21185 -22.89916),POINT(-43.21185 -22.89916),1d892d29bfad4afa8edde52f016c57f99454e9e5
1,1,24049,Intersul,NaN,regular,006j,0,Circular,LECD130,24049.649573,U,2026-03-14,2026-03-14,2026-03-27,"LINESTRING(-43.21592 -22.9801, -43.21543 -22.9...",24111.3,POINT(-43.21185 -22.89916),POINT(-43.21185 -22.89916),3a6c7f1b4d7d9c5cc07568b119f9e07d65ab5cb7
2,1,24049,Intersul,NaN,regular,006j,0,Circular,LECD130,24049.649573,U,2026-02-28,2026-02-28,2026-03-04,"LINESTRING(-43.21592 -22.9801, -43.21543 -22.9...",24111.3,POINT(-43.21185 -22.89916),POINT(-43.21185 -22.89916),1d892d29bfad4afa8edde52f016c57f99454e9e5
3,2,8502,Santa Cruz,NaN,regular,01gk,1,Jardim Bangu,819,8502.547401,U,2024-12-31,2024-12-31,2025-01-01,"LINESTRING(-43.46371 -22.87638, -43.46372 -22....",8513.6,POINT(-43.46371 -22.87638),POINT(-43.4676 -22.84347),3e1630ddb15dca783e340faf23f08f68733f4813
4,2,8502,Santa Cruz,NaN,regular,01gk,1,Jardim Bangu,819,8502.547401,U,2024-06-17,2024-06-17,2024-06-30,"LINESTRING(-43.46371 -22.87638, -43.46372 -22....",8513.6,POINT(-43.46371 -22.87638),POINT(-43.4676 -22.84347),f9af0818a672ad3ef560f78d9f425175d88f75bf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9473,952,10374,Transcarioca,NaN,regular,ztj0,0,Praça Seca,783,10374.501892,U,2023-10-16,2023-10-16,2023-10-16,"LINESTRING(-43.38416 -22.854038, -43.38416 -22...",10404.8,POINT(-43.38416 -22.854038),POINT(-43.34843 -22.89695),201d79faee763526a030ff998bebea9782efe961
9474,952,10374,Transcarioca,NaN,regular,ztj0,0,Praça Seca,783,10374.501892,U,2026-02-02,2026-02-02,2026-02-10,"LINESTRING(-43.384006 -22.854102, -43.38371 -2...",10387.4,POINT(-43.384006 -22.854102),POINT(-43.34843 -22.89695),f7495a359559836032529b2d93bc4eff1a079265
9475,952,10374,Transcarioca,NaN,regular,ztj0,0,Praça Seca,783,10374.501892,U,2025-05-01,2025-05-01,2025-05-23,"LINESTRING(-43.384006 -22.854102, -43.38371 -2...",10387.4,POINT(-43.384006 -22.854102),POINT(-43.34843 -22.89695),122abe77b87c75b8505fc679ef662556336ca16f
9476,953,30066,Transcarioca,NaN,regular,zycy,1,Terminal Alvorada,LECD129,30066.291260,U,2026-02-28,2026-02-28,2026-03-04,"LINESTRING(-43.19232 -22.90502, -43.19233 -22....",30079.0,POINT(-43.19232 -22.90502),POINT(-43.36692 -23.00162),1d892d29bfad4afa8edde52f016c57f99454e9e5


### Tarefa #2: Pegando um ponto saber se está dentro de uma forma do geo

In [ ]:
cisp_df = gpd.read_file('/content/lm_cisp_bd.dbf')

In [ ]:
cisp_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 137 entries, 0 to 136
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   cisp        137 non-null    int64   
 1   aisp        137 non-null    int64   
 2   shape_Leng  137 non-null    float64 
 3   shape_Area  137 non-null    float64 
 4   AREA_GEO    137 non-null    float64 
 5   geometry    136 non-null    geometry
dtypes: float64(3), geometry(1), int64(2)
memory usage: 6.6 KB


In [ ]:
cisp_df.head()

,cisp,aisp,shape_Leng,shape_Area,AREA_GEO,geometry
0,120,35,1.595433,0.082321,9.375473e+08,"POLYGON ((-42.18687 -22.55548, -42.18733 -22.5..."
1,127,25,0.697487,0.006180,7.027806e+07,"MULTIPOLYGON (((-41.90082 -22.78264, -41.90079..."
2,151,11,1.752923,0.081815,9.334145e+08,"POLYGON ((-42.52555 -22.16087, -42.5006 -22.18..."
3,107,38,1.209676,0.050835,5.805239e+08,"POLYGON ((-43.34665 -22.00375, -43.34603 -22.0..."
4,123,32,2.118860,0.106640,1.216847e+09,"MULTIPOLYGON (((-41.97224 -22.1428, -41.97201 ..."


In [ ]:
from shapely.geometry import Point

def get_cisp_from_point(latitude, longitude):
    """
    Determina a qual CISP um ponto geográfico pertence.

    Args:
        latitude (float): A latitude do ponto.
        longitude (float): A longitude do ponto.

    Returns:
        int or None: O ID da CISP à qual o ponto pertence, ou None se não pertencer a nenhuma.
    """
    # Cria um objeto Point a partir das coordenadas
    point = Point(longitude, latitude)

    # Itera sobre cada CISP no GeoDataFrame
    for index, row in cisp_df.iterrows():
        if row['geometry'] is not None and row['geometry'].contains(point):
            return row['cisp']
    return None

# Exemplo de uso da função com um ponto de teste

sample_latitude = -22.898278 # Exemplo de latitude
sample_longitude = -43.760250 # Exemplo de longitude

cisp_id = get_cisp_from_point(sample_latitude, sample_longitude)

if cisp_id:
    print(f"O ponto ({sample_latitude}, {sample_longitude}) pertence à CISP: {cisp_id}")
else:
    print(f"O ponto ({sample_latitude}, {sample_longitude}) não pertence a nenhuma CISP conhecida.")

O ponto (-22.898278, -43.76025) pertence à CISP: 36


### Tarefa #3:  Criar as listas com as CISPs para cada linha


In [ ]:
servico_shape_df = shape_e_itinerarios_df[['servico', 'shape_id', 'shape']]
print("DataFrame com as colunas 'servico', 'shape_id' e 'shape':")
servico_shape_df.head()

DataFrame com as colunas 'servico', 'shape_id' e 'shape':


,servico,shape_id,shape
0,LECD130,006j,"LINESTRING(-43.21592 -22.9801, -43.21543 -22.9..."
1,LECD130,006j,"LINESTRING(-43.21592 -22.9801, -43.21543 -22.9..."
2,LECD130,006j,"LINESTRING(-43.21592 -22.9801, -43.21543 -22.9..."
3,819,01gk,"LINESTRING(-43.46371 -22.87638, -43.46372 -22...."
4,819,01gk,"LINESTRING(-43.46371 -22.87638, -43.46372 -22...."


In [ ]:
servico_shape_df.head()

,servico,shape_id,shape
0,LECD130,006j,"LINESTRING(-43.21592 -22.9801, -43.21543 -22.9..."
1,LECD130,006j,"LINESTRING(-43.21592 -22.9801, -43.21543 -22.9..."
2,LECD130,006j,"LINESTRING(-43.21592 -22.9801, -43.21543 -22.9..."
3,819,01gk,"LINESTRING(-43.46371 -22.87638, -43.46372 -22...."
4,819,01gk,"LINESTRING(-43.46371 -22.87638, -43.46372 -22...."


In [ ]:
servico_shape_df[servico_shape_df['servico'] == '487']

,servico,shape_id,shape


In [ ]:
from shapely import wkt

def get_all_cisps_for_route_shape(shape_wkt):
    """
    Extrai todas as CISPs por onde uma rota de ônibus passa, dada a geometria da rota.

    Args:
        shape_wkt (str): A representação WKT (Well-Known Text) da geometria da rota (LINESTRING ou MULTILINESTRING).

    Returns:
        list: Uma lista ordenada e única de IDs de CISPs que a rota atravessa.
    """
    geometry_obj = wkt.loads(shape_wkt)
    cisp_ids_for_route = set()

    # Helper function to extract points from any geometry type
    def extract_points(geometry):
        if geometry.geom_type == 'Point':
            yield geometry
        elif geometry.geom_type == 'LineString':
            for coord in geometry.coords:
                yield Point(coord)
        elif geometry.geom_type == 'MultiPoint':
            for point in geometry.geoms:
                yield point
        elif geometry.geom_type == 'MultiLineString':
            for line in geometry.geoms:
                for coord in line.coords:
                    yield Point(coord)
        elif geometry.geom_type == 'Polygon':
            # For a polygon, we're interested in the boundary for a route
            for coord in geometry.exterior.coords:
                yield Point(coord)
        elif geometry.geom_type == 'MultiPolygon':
            for poly in geometry.geoms:
                for coord in poly.exterior.coords:
                    yield Point(coord)

    for point_obj in extract_points(geometry_obj):
        # get_cisp_from_point expects latitude, longitude. Shapely Point is (longitude, latitude)
        cisp_id = get_cisp_from_point(point_obj.y, point_obj.x)
        if cisp_id is not None:
            cisp_ids_for_route.add(cisp_id)

    return sorted(list(cisp_ids_for_route))

# Pegar a primeira linha do DataFrame servico_shape_df
first_row = servico_shape_df.iloc[8]
first_service_id = first_row['servico']
first_shape_wkt = first_row['shape']

# Gerar a lista de CISPs para essa rota
cisps_for_first_service = get_all_cisps_for_route_shape(first_shape_wkt)

print(f"O serviço '{first_service_id}' passa pelas seguintes CISPs: {cisps_for_first_service}")

O serviço '819' passa pelas seguintes CISPs: [34]


In [ ]:
# Crie uma coluna 'cisp_ids' aplicando a função a cada 'shape' em servico_shape_df
# Para evitar processar formas duplicadas, vamos agrupar por 'shape_id' e aplicar a função uma vez por forma única.
# Em seguida, juntamos os resultados de volta ao DataFrame original.

unique_shapes = servico_shape_df[['shape_id', 'shape']].drop_duplicates(subset=['shape_id'])

def process_shape_for_cisps(row):
    try:
        return get_all_cisps_for_route_shape(row['shape'])
    except Exception as e:
        # Adiciona um tratamento de erro para shapes inválidos
        print(f"Erro ao processar shape_id {row['shape_id']}: {e}")
        return []

cisp_per_shape = unique_shapes.copy()
cisp_per_shape['cisp_ids'] = cisp_per_shape.apply(process_shape_for_cisps, axis=1)

# Agora, mescle cisp_per_shape com itinerarios_df para adicionar as CISPs aos serviços
# Usamos o 'shape_id' para fazer o merge e depois agrupamos por 'servico'
itinerarios_with_cisps_details = itinerarios_df.merge(
    cisp_per_shape[['shape_id', 'cisp_ids']],
    on='shape_id',
    how='left'
)

# Agrega as cisp_ids para cada 'servico' único, remove duplicatas e ordena
service_cisps_aggregated = itinerarios_with_cisps_details.groupby('servico')['cisp_ids'].apply(
    lambda x: sorted(list(set(sum((item for item in x if isinstance(item, list) and item is not None), []))))
).reset_index()

service_cisps_aggregated.rename(columns={'cisp_ids': 'cisp_list'}, inplace=True)

print("DataFrame com a lista de CISPs para cada serviço de ônibus:")
display(service_cisps_aggregated.head())

# Exemplo para um serviço específico
if not service_cisps_aggregated.empty:
    example_service = service_cisps_aggregated.iloc[0]
    print(f"\nExemplo para o serviço '{example_service['servico']}':")
    print(f"CISPs associadas: {example_service['cisp_list']}")
else:
    print("Não foi possível encontrar nenhum serviço de ônibus para exemplificar.")

DataFrame com a lista de CISPs para cada serviço de ônibus:


,servico,cisp_list
0,006,"[5, 7, 9]"
1,007,"[1, 4, 5, 7, 9]"
2,010,"[1, 4, 5, 9]"
3,10,"[16, 36, 42, 43]"
4,100,"[1, 4, 5, 9, 10, 12, 13, 14]"



Exemplo para o serviço '006':
CISPs associadas: [5, 7, 9]


In [ ]:
service_cisps_aggregated['cisp_list']

,cisp_list
0,"[5, 7, 9]"
1,"[1, 4, 5, 7, 9]"
2,"[1, 4, 5, 9]"
3,"[16, 36, 42, 43]"
4,"[1, 4, 5, 9, 10, 12, 13, 14]"
...,...
486,"[21, 22, 23, 24, 28, 29, 30, 33, 34, 44]"
487,[]
488,[]
489,"[17, 18, 19, 21, 22, 27, 38, 39, 40]"
